In [1]:
import pandas as pd
import numpy as np
import torch

from pathlib import Path


PROJECT_DIR = Path("/workspace/primekg_ddi_rgcn")

GRAPH_DIR = (
    PROJECT_DIR
    / "data/processed/graphs"
)

TENSOR_DIR = (
    PROJECT_DIR
    / "data/processed/rgcn_tensors"
)

MAPPING_DIR = (
    PROJECT_DIR
    / "data/processed/mappings"
)


print("Project:", PROJECT_DIR)
print("Graph dir:", GRAPH_DIR)
print("Tensor dir:", TENSOR_DIR)


Project: /workspace/primekg_ddi_rgcn
Graph dir: /workspace/primekg_ddi_rgcn/data/processed/graphs
Tensor dir: /workspace/primekg_ddi_rgcn/data/processed/rgcn_tensors


In [2]:
g0 = pd.read_parquet(
    GRAPH_DIR / "G0_edges.parquet"
)

g3 = pd.read_parquet(
    GRAPH_DIR / "G3_edges.parquet"
)

print("G0 edges:", len(g0))
print("G3 edges:", len(g3))

print("\nG3 relation counts:")
print(
    g3["relation"]
    .value_counts()
)


G0 edges: 2138160
G3 edges: 2274728

G3 relation counts:
relation
drug_drug               2138160
rev_contraindication      30675
contraindication          30675
target                    16380
rev_target                16380
indication                 9388
rev_indication             9388
enzyme                     5317
rev_enzyme                 5317
transporter                3092
rev_transporter            3092
off-label use              2568
rev_off-label use          2568
rev_carrier                 864
carrier                     864
Name: count, dtype: int64


In [3]:
ABLATIONS = {

    "A1_target": [
        "target",
        "rev_target"
    ],

    "A2_enzyme": [
        "enzyme",
        "rev_enzyme"
    ],

    "A3_transporter": [
        "transporter",
        "rev_transporter"
    ],

    "A4_carrier": [
        "carrier",
        "rev_carrier"
    ],

    "A5_indication": [
        "indication",
        "rev_indication"
    ],

    "A6_contraindication": [
        "contraindication",
        "rev_contraindication"
    ],

    "A7_off_label": [
        "off-label use",
        "rev_off-label use"
    ],
}


In [4]:
ablation_graphs = {}

for experiment_name, support_relations in ABLATIONS.items():

    support_edges = g3[
        g3["relation"].isin(
            support_relations
        )
    ].copy()

    ablation_edges = pd.concat(
        [
            g0,
            support_edges
        ],
        ignore_index=True
    )

    ablation_graphs[
        experiment_name
    ] = ablation_edges

    output_path = (
        GRAPH_DIR
        / f"{experiment_name}_edges.parquet"
    )

    ablation_edges.to_parquet(
        output_path,
        index=False
    )

    print("\n========================")
    print(experiment_name)
    print("========================")

    print(
        "Directed edges:",
        f"{len(ablation_edges):,}"
    )

    print(
        "Relations:",
        sorted(
            ablation_edges[
                "relation"
            ].unique()
        )
    )

    print("\nCounts:")

    print(
        ablation_edges[
            "relation"
        ].value_counts()
    )

    print("\nSaved:")
    print(output_path)


A1_target
Directed edges: 2,170,920
Relations: ['drug_drug', 'rev_target', 'target']

Counts:
relation
drug_drug     2138160
target          16380
rev_target      16380
Name: count, dtype: int64

Saved:
/workspace/primekg_ddi_rgcn/data/processed/graphs/A1_target_edges.parquet

A2_enzyme
Directed edges: 2,148,794
Relations: ['drug_drug', 'enzyme', 'rev_enzyme']

Counts:
relation
drug_drug     2138160
enzyme           5317
rev_enzyme       5317
Name: count, dtype: int64

Saved:
/workspace/primekg_ddi_rgcn/data/processed/graphs/A2_enzyme_edges.parquet

A3_transporter
Directed edges: 2,144,344
Relations: ['drug_drug', 'rev_transporter', 'transporter']

Counts:
relation
drug_drug          2138160
transporter           3092
rev_transporter       3092
Name: count, dtype: int64

Saved:
/workspace/primekg_ddi_rgcn/data/processed/graphs/A3_transporter_edges.parquet

A4_carrier
Directed edges: 2,139,888
Relations: ['carrier', 'drug_drug', 'rev_carrier']

Counts:
relation
drug_drug      2138160
c

In [5]:
# ============================================================
# Convert relation-ablation graphs to R-GCN PyTorch tensors
# ============================================================

import pandas as pd
import numpy as np
import torch
from pathlib import Path


PROJECT_DIR = Path("/workspace/primekg_ddi_rgcn")

GRAPH_DIR = PROJECT_DIR / "data/processed/graphs"
TENSOR_DIR = PROJECT_DIR / "data/processed/rgcn_tensors"
MAPPING_DIR = PROJECT_DIR / "data/processed/mappings"

TENSOR_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Load the SAME mappings used by G0-G3
# ------------------------------------------------------------

node_mapping = pd.read_parquet(
    MAPPING_DIR / "node_mapping.parquet"
)

relation_mapping = pd.read_parquet(
    MAPPING_DIR / "relation_mapping.parquet"
)


print("Node mapping rows:", len(node_mapping))
print("Relation mapping rows:", len(relation_mapping))

print("\nRelation mapping:")
print(relation_mapping)


# ------------------------------------------------------------
# 2. Create lookup dictionaries
# ------------------------------------------------------------

node_id_map = dict(
    zip(
        node_mapping["original_index"],
        node_mapping["node_id"]
    )
)

relation_id_map = dict(
    zip(
        relation_mapping["relation"],
        relation_mapping["relation_id"]
    )
)


# ------------------------------------------------------------
# 3. Ablation experiment names
# ------------------------------------------------------------

ABLATION_NAMES = [
    "A1_target",
    "A2_enzyme",
    "A3_transporter",
    "A4_carrier",
    "A5_indication",
    "A6_contraindication",
    "A7_off_label",
]


# ------------------------------------------------------------
# 4. Convert each graph
# ------------------------------------------------------------

for experiment_name in ABLATION_NAMES:

    print("\n")
    print("=" * 60)
    print(experiment_name)
    print("=" * 60)

    graph_path = (
        GRAPH_DIR
        / f"{experiment_name}_edges.parquet"
    )

    graph = pd.read_parquet(graph_path)

    print(
        "Loaded directed edges:",
        f"{len(graph):,}"
    )


    # --------------------------------------------------------
    # Convert original node indices -> global R-GCN node IDs
    # --------------------------------------------------------

    src_ids = graph["src_index"].map(node_id_map)
    dst_ids = graph["dst_index"].map(node_id_map)


    # --------------------------------------------------------
    # Convert relation names -> global relation IDs
    # --------------------------------------------------------

    rel_ids = graph["relation"].map(
        relation_id_map
    )


    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert src_ids.notna().all(), (
        f"{experiment_name}: "
        "Some source nodes are missing from node mapping."
    )

    assert dst_ids.notna().all(), (
        f"{experiment_name}: "
        "Some destination nodes are missing from node mapping."
    )

    assert rel_ids.notna().all(), (
        f"{experiment_name}: "
        "Some relations are missing from relation mapping."
    )


    # --------------------------------------------------------
    # Create edge_index
    #
    # Shape:
    # [2, number_of_edges]
    # --------------------------------------------------------

    edge_index = torch.tensor(
        np.vstack(
            [
                src_ids.to_numpy(
                    dtype=np.int64
                ),
                dst_ids.to_numpy(
                    dtype=np.int64
                )
            ]
        ),
        dtype=torch.long
    )


    # --------------------------------------------------------
    # Create edge_type
    # --------------------------------------------------------

    edge_type = torch.tensor(
        rel_ids.to_numpy(
            dtype=np.int64
        ),
        dtype=torch.long
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Keep num_relations = 15.
    #
    # We are using the SAME global relation space as G0-G3.
    # --------------------------------------------------------

    tensor_data = {

        "edge_index": edge_index,

        "edge_type": edge_type,

        "num_nodes": len(node_mapping),

        "num_relations": len(relation_mapping),

        "experiment": experiment_name,

        "active_relation_ids": torch.unique(
            edge_type
        ).sort().values
    }


    # --------------------------------------------------------
    # Save tensor
    # --------------------------------------------------------

    tensor_path = (
        TENSOR_DIR
        / f"{experiment_name}.pt"
    )

    torch.save(
        tensor_data,
        tensor_path
    )


    # --------------------------------------------------------
    # Print verification information
    # --------------------------------------------------------

    active_ids = torch.unique(
        edge_type
    ).sort().values.tolist()


    print(
        "edge_index shape:",
        edge_index.shape
    )

    print(
        "edge_type shape:",
        edge_type.shape
    )

    print(
        "num_nodes:",
        tensor_data["num_nodes"]
    )

    print(
        "num_relations:",
        tensor_data["num_relations"]
    )

    print(
        "Active relation IDs:",
        active_ids
    )

    print(
        "Saved:",
        tensor_path
    )


print("\n")
print("=" * 60)
print("ALL ABLATION TENSORS CREATED")
print("=" * 60)

Node mapping rows: 13094
Relation mapping rows: 15

Relation mapping:
                relation  relation_id
0              drug_drug            0
1                 target            1
2             rev_target            2
3                 enzyme            3
4             rev_enzyme            4
5            transporter            5
6        rev_transporter            6
7                carrier            7
8            rev_carrier            8
9             indication            9
10        rev_indication           10
11      contraindication           11
12  rev_contraindication           12
13         off-label use           13
14     rev_off-label use           14


A1_target
Loaded directed edges: 2,170,920
edge_index shape: torch.Size([2, 2170920])
edge_type shape: torch.Size([2170920])
num_nodes: 13094
num_relations: 15
Active relation IDs: [0, 1, 2]
Saved: /workspace/primekg_ddi_rgcn/data/processed/rgcn_tensors/A1_target.pt


A2_enzyme
Loaded directed edges: 2,148,794
edge_ind